# Verne, small, step 04: the sampler (roadmap box 05), on T1-wide

No explained notebook for this step: nothing is trained, and the one new formula is the τ grid below. Step 03: `03_vae_latents.ipynb`.

One box opened, inference only: how the action chunk is denoised. Steps 0-03 took 10 even Euler steps from noise (τ = 0) to the chunk (τ = 1). Verne (`../verne-v0.5/src/verne/model/sampler.py`, test config) takes 4 steps on a shifted grid: noise level σ = 1 − τ, σ(u) = s·u / (1 + (s − 1)·u) on u even from 1 to 0, s = 1.5 (`sample` in `model.py`, derivation above it). The policies are step 03's three T1-wide runs, unchanged; only the sampler differs.

Runs: `experiments/verne/scripts/sampler_sweep.py` → `experiments/verne/runs/04_sampler/<arm>/t1-wide/seed<k>/`, all on the local RTX 4060 (step 03's own sampler rerun here first: same scores as on the L40S).

In [ ]:
import json

import torch

from dream_robot.policies.verne.run import RUNS

ARMS = ('s10_shift1.0', 's4_shift1.0', 's4_shift1.5', 's10_shift1.5')
LABEL = {
    's10_shift1.0': '10 even (steps 0-03)',
    's4_shift1.0': '4 even',
    's4_shift1.5': '4, shift 1.5 (Verne)',
    's10_shift1.5': '10, shift 1.5',
}

### step 04.1 — the τ grids

Where each sampler evaluates the velocity, from noise (0) to the chunk (1):

In [ ]:
for arm in ARMS:
    steps, shift = int(arm[1:].split('_')[0]), float(arm.split('shift')[1])
    u = torch.linspace(1, 0, steps + 1)
    tau = 1 - shift * u / (1 + (shift - 1) * u)
    print(f'{LABEL[arm]:22s} τ = {" ".join(f"{t:.2f}" for t in tau.tolist())}')

10 even (steps 0-03)   τ = 0.00 0.10 0.20 0.30 0.40 0.50 0.60 0.70 0.80 0.90 1.00
4 even                 τ = 0.00 0.25 0.50 0.75 1.00
4, shift 1.5 (Verne)   τ = 0.00 0.18 0.40 0.67 1.00
10, shift 1.5          τ = 0.00 0.07 0.14 0.22 0.31 0.40 0.50 0.61 0.73 0.86 1.00


### step 04.2 — T1-wide, 3 training seeds × 50 rollout seeds, each sampler

In [ ]:
def results(arm: str):
    # (training seed, n_exec, rollout seed) -> success, for one sampler arm
    out = {}
    for seed in (0, 1, 2):
        for n in (8, 16):
            path = RUNS / '04_sampler' / arm / 't1-wide' / f'seed{seed}' / f'sweep_{n}.json'
            out |= {(seed, n, e): r[0] for e, r in json.loads(path.read_text()).items()}
    return out


res = {arm: results(arm) for arm in ARMS}

In [ ]:
base = res['s10_shift1.0']
print(f'{"sampler":22s} {"n_exec 8":>18s} {"n_exec 16":>18s}   vs 10 even: won / lost of 300')
for arm in ARMS:
    r = res[arm]
    cols = []
    for n in (8, 16):
        per = [sum(r[s, n, e] for (s, m, e) in r if s == seed and m == n) for seed in (0, 1, 2)]
        cols.append(f'{"+".join(map(str, per))} = {sum(per)}/150')
    won = sum(r[k] and not base[k] for k in base)
    lost = sum(base[k] and not r[k] for k in base)
    print(f'{LABEL[arm]:22s} {cols[0]:>18s} {cols[1]:>18s}   {won:3d} / {lost}')

sampler                          n_exec 8          n_exec 16   vs 10 even: won / lost of 300
10 even (steps 0-03)   34+36+42 = 112/150 36+37+39 = 112/150     0 / 0
4 even                 33+41+44 = 118/150 37+40+43 = 120/150    17 / 3
4, shift 1.5 (Verne)   36+39+45 = 120/150 39+38+45 = 122/150    23 / 5
10, shift 1.5          31+37+42 = 110/150 36+38+40 = 114/150     6 / 6


### wrap-up — the sampler (box 05): Verne's 4 shifted steps, now the default

**What it buys:** 4 steps instead of 10 win on the same episodes: 17 won / 3 lost with even steps, 23 / 5 with Verne's shift 1.5 (120 and 122 of 150 against 112 and 112), and the action side runs 4 times per replan instead of 10. `sample` now defaults to (4, 1.5), so later steps sample Verne's way.

**Where it fails:** the shift on its own does nothing (10 shifted steps: 6 won / 6 lost); on top of 4 steps it adds 2 per 150, within noise. Why fewer steps help is a guess, not tested: bigger steps may land each chunk nearer the average action, steadier for a precise grasp than 10 small steps that can drift. The grasp failures (`no_grasp`) are fewer, not gone.

**The grasp tightened a little** (close-point probe, `scripts/sampler_probe.py`, n_exec 8, 3 seeds × 50, same episodes): mean miss at the first close 1.38 → 1.25 cm (seed 1: 1.7 → 1.4), successes 1.06 → 0.97 cm. Failures still miss by ~2.3 cm, mostly sideways (|dy| 2.0), and the share of closes more than 2 cm off barely moves (16% → 15%); `no_grasp` falls 69 → 50 over both n_exec. So 4 steps pulls borderline closes in, it doesn't fix the bad ones.

**Untested:** other step counts (2, 6), and how this interacts with training: a model trained for a different time embedding (box 04) might prefer another grid.